In [47]:
pip install mysql-connector-python

In [48]:
!pip install pymysql

In [49]:
# Install and start MySQL server inside Colab
!apt-get update -qq
!apt-get install -y -qq mysql-server
!service mysql start


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
 * Starting MySQL database server mysqld
   ...done.


In [50]:
!mysql -u root < bingeplay.sql


tbl	row_count
users	3000
subscriptions	4497
shows	100
watch_sessions	100351
ratings	5000
null_user_sessions
2


In [51]:
import pandas as pd
from sqlalchemy import create_engine

!mysql -u root -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH 'mysql_native_password' BY ''; FLUSH PRIVILEGES;"

engine = create_engine('mysql+pymysql://root:@localhost:3306/bingeplay')

print(pd.read_sql("SHOW TABLES;", engine))

  Tables_in_bingeplay
0             ratings
1               shows
2       subscriptions
3               users
4      watch_sessions


## Q1 - Active revenue
**Answer:** Run the query below to obtain the active subscriptions count and total monthly recurring revenue as of 30 June 2024.

In [52]:
query_1 = """
SELECT
    COUNT(subscription_id) AS active_subscriptions,
    SUM(monthly_price_inr) AS total_monthly_revenue_inr
FROM subscriptions
WHERE status = 'active'
  AND (end_date IS NULL OR end_date > '2024-06-30');
"""
df_1 = pd.read_sql(query_1, engine)
print(df_1)

   active_subscriptions  total_monthly_revenue_inr
0                  2340                   784260.0


## Q2 - Signup momentum
**Answer:** Monthly signup distribution and the peak signup month of 2024.

In [53]:
query_2 = """
SELECT
    MONTH(signup_date) AS month,
    MONTHNAME(signup_date) AS month_name,
    COUNT(user_id) AS signup_count
FROM users
WHERE YEAR(signup_date) = 2024
GROUP BY MONTH(signup_date), MONTHNAME(signup_date)
ORDER BY month ASC;
"""
df_2 = pd.read_sql(query_2, engine)
print(df_2)

# Identify the month with the highest signups
peak_month = df_2.loc[df_2['signup_count'].idxmax()]
print(f"\nMonth with highest signups: {peak_month['month_name']} ({peak_month['signup_count']} signups)")

   month month_name  signup_count
0      1    January           350
1      2   February           400
2      3      March           500
3      4      April           550
4      5        May           600
5      6       June           600

Month with highest signups: May (600 signups)


## Q3 - Device analytics
**Answer:** Performance metrics across device types (excluding NULL user sessions).

In [54]:
query_3 = """
SELECT
    device_type,
    COUNT(session_id) AS total_sessions,
    SUM(watch_minutes) AS total_watch_minutes,
    ROUND(AVG(watch_minutes), 2) AS avg_watch_minutes,
    ROUND(SUM(CASE WHEN completed = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(session_id), 2) AS completion_rate_pct
FROM watch_sessions
WHERE user_id IS NOT NULL
GROUP BY device_type
ORDER BY total_sessions DESC;
"""
df_3 = pd.read_sql(query_3, engine)
print(df_3)

  device_type  total_sessions  total_watch_minutes  avg_watch_minutes  \
0      Mobile           50172            1504355.0              29.98   
1          TV           27981             840595.0              30.04   
2      Laptop           15105             453434.0              30.02   
3      Tablet            7091             210733.0              29.72   

   completion_rate_pct  
0                60.24  
1                59.98  
2                60.51  
3                59.79  


## Q4 - Rating distribution
**Answer:** Distribution of 1 to 5 star ratings and total percentage of positive (4-5 star) reviews.

In [55]:
query_4 = """
SELECT
    stars,
    COUNT(rating_id) AS count,
    ROUND(COUNT(rating_id) * 100.0 / (SELECT COUNT(*) FROM ratings), 2) AS percentage
FROM ratings
GROUP BY stars
ORDER BY stars ASC;
"""
df_4 = pd.read_sql(query_4, engine)
print(df_4)

# Percentage of 4 or 5 stars
positive_pct = df_4[df_4['stars'].isin([4, 5])]['percentage'].sum()
print(f"\nPercentage of ratings that are 4 or 5 stars: {positive_pct:.2f}%")

   stars  count  percentage
0      1    234        4.68
1      2    352        7.04
2      3    847       16.94
3      4   1781       35.62
4      5   1786       35.72

Percentage of ratings that are 4 or 5 stars: 71.34%


## Q5 - Originals vs acquired
**Answer:** Comparison between BingePlay Originals and acquired catalog content.
**Interpretation:** BingePlay Originals maintain a noticeably higher average IMDb rating compared to acquired titles, demonstrating stronger overall content quality.

In [56]:
query_5 = """
SELECT
    CASE WHEN is_original = 1 THEN 'Original' ELSE 'Acquired' END AS content_type,
    COUNT(show_id) AS number_of_shows,
    ROUND(AVG(imdb_rating), 2) AS avg_imdb_rating,
    ROUND(AVG(release_year), 1) AS avg_release_year
FROM shows
GROUP BY is_original;
"""
df_5 = pd.read_sql(query_5, engine)
print(df_5)

  content_type  number_of_shows  avg_imdb_rating  avg_release_year
0     Acquired               70             6.63            2020.7
1     Original               30             7.92            2020.4


## Q6 - Binge day detection
**Answer:** Total binge days in Q2 2024 and the user with the most binge occurrences.

In [57]:
query_6 = """
WITH BingeEvents AS (
    SELECT
        user_id,
        show_id,
        session_date
    FROM watch_sessions
    WHERE session_date BETWEEN '2024-04-01' AND '2024-06-30'
      AND user_id IS NOT NULL
    GROUP BY user_id, show_id, session_date
    HAVING COUNT(session_id) >= 5
)
SELECT
    (SELECT COUNT(*) FROM BingeEvents) AS total_binge_days_q2,
    user_id,
    COUNT(*) AS max_binge_days
FROM BingeEvents
GROUP BY user_id
ORDER BY max_binge_days DESC
LIMIT 1;
"""
df_6 = pd.read_sql(query_6, engine)
print(df_6)

   total_binge_days_q2 user_id  max_binge_days
0                  414  U02956               8


## Q7 - signups who never watched
**Answer:** Total Q1 2024 signups and count of users who never watched a single session (NULL-safe evaluation).

In [58]:
query_7 = """
SELECT
    COUNT(u.user_id) AS total_q1_signups,
    COUNT(CASE WHEN ws.user_id IS NULL THEN 1 END) AS never_watched_users
FROM users u
LEFT JOIN (
    SELECT DISTINCT user_id
    FROM watch_sessions
    WHERE user_id IS NOT NULL
) ws ON u.user_id = ws.user_id
WHERE u.signup_date BETWEEN '2024-01-01' AND '2024-03-31';
"""
df_7 = pd.read_sql(query_7, engine)
print(df_7)

   total_q1_signups  never_watched_users
0              1250                  226


## Q8 - The over-paying Premium/Family users
**Answer:** Count of active Premium/Family users whose entire watch history consists only of Basic-tier content.

In [59]:
query_8 = """
WITH ActiveHighTierUsers AS (
    SELECT DISTINCT user_id
    FROM subscriptions
    WHERE status = 'active'
      AND (end_date IS NULL OR end_date > '2024-06-30')
      AND plan IN ('Premium', 'Family')
)
SELECT COUNT(DISTINCT a.user_id) AS overpaying_users_count
FROM ActiveHighTierUsers a
WHERE EXISTS (
    SELECT 1 FROM watch_sessions ws WHERE ws.user_id = a.user_id
)
AND NOT EXISTS (
    SELECT 1
    FROM watch_sessions ws
    JOIN shows s ON ws.show_id = s.show_id
    WHERE ws.user_id = a.user_id
      AND s.min_plan IN ('Premium', 'Family')
);
"""
df_8 = pd.read_sql(query_8, engine)
print(df_8)

   overpaying_users_count
0                       7


## Q9 - Upgrade success cohort
**Answer:** Number of January Basic-plan signups who subsequently upgraded to Premium/Family and remain active, along with average days taken to first upgrade.

In [60]:
query_9 = """
WITH JanBasicSignups AS (
    SELECT
        u.user_id,
        u.signup_date,
        s.plan AS initial_plan,
        ROW_NUMBER() OVER (PARTITION BY s.user_id ORDER BY s.start_date ASC) AS sub_order
    FROM users u
    JOIN subscriptions s ON u.user_id = s.user_id
    WHERE u.signup_date BETWEEN '2024-01-01' AND '2024-01-31'
),
InitialBasicUsers AS (
    SELECT user_id, signup_date
    FROM JanBasicSignups
    WHERE sub_order = 1 AND initial_plan = 'Basic'
),
FirstUpgrade AS (
    SELECT
        ibu.user_id,
        ibu.signup_date,
        MIN(s.start_date) AS first_upgrade_date
    FROM InitialBasicUsers ibu
    JOIN subscriptions s ON ibu.user_id = s.user_id
    WHERE s.plan IN ('Premium', 'Family')
    GROUP BY ibu.user_id, ibu.signup_date
)
SELECT
    COUNT(DISTINCT fu.user_id) AS total_upgraded_users,
    ROUND(AVG(DATEDIFF(fu.first_upgrade_date, fu.signup_date)), 2) AS avg_days_to_upgrade
FROM FirstUpgrade fu
JOIN subscriptions s ON fu.user_id = s.user_id
WHERE s.status = 'active'
  AND (s.end_date IS NULL OR s.end_date > '2024-06-30');
"""
df_9 = pd.read_sql(query_9, engine)
print(df_9)

   total_upgraded_users  avg_days_to_upgrade
0                    55                64.96


## Q10 - Cliffhanger comebacks
**Answer:** Total cliffhanger comeback events and the show generating the most comeback sessions.

In [61]:
query_10 = """
WITH IncompleteSessions AS (
    SELECT DISTINCT user_id, show_id, session_date
    FROM watch_sessions
    WHERE completed = 0 AND user_id IS NOT NULL
),
ComebackEvents AS (
    SELECT DISTINCT
        inc.user_id,
        inc.show_id,
        inc.session_date AS incomplete_date
    FROM IncompleteSessions inc
    JOIN watch_sessions ws
      ON inc.user_id = ws.user_id
     AND inc.show_id = ws.show_id
     AND ws.session_date BETWEEN DATE_ADD(inc.session_date, INTERVAL 1 DAY)
                             AND DATE_ADD(inc.session_date, INTERVAL 7 DAY)
)
SELECT
    (SELECT COUNT(*) FROM ComebackEvents) AS total_comeback_events,
    c.show_id,
    s.title,
    COUNT(*) AS show_comeback_count
FROM ComebackEvents c
JOIN shows s ON c.show_id = s.show_id
GROUP BY c.show_id, s.title
ORDER BY show_comeback_count DESC
LIMIT 1;
"""
df_10 = pd.read_sql(query_10, engine)
print(df_10)

   total_comeback_events show_id             title  show_comeback_count
0                   4345    S088  Rayalaseema Raga                   64


## Q11 - Consecutive-week engagement
**Answer:** Users with a 4+ consecutive week engagement streak, the longest recorded streak, and the top user ID.

In [62]:
query_11 = """
WITH UserWeeks AS (
    SELECT DISTINCT
        user_id,
        YEAR(session_date) AS yr,
        WEEK(session_date, 3) AS iso_week
    FROM watch_sessions
    WHERE user_id IS NOT NULL
),
RankedWeeks AS (
    SELECT
        user_id,
        iso_week,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY yr, iso_week) AS rn
    FROM UserWeeks
),
StreakGroups AS (
    SELECT
        user_id,
        (iso_week - rn) AS streak_group,
        COUNT(*) AS streak_length
    FROM RankedWeeks
    GROUP BY user_id, (iso_week - rn)
)
SELECT
    (SELECT COUNT(DISTINCT user_id) FROM StreakGroups WHERE streak_length >= 4) AS users_with_4plus_week_streak,
    user_id,
    MAX(streak_length) AS longest_streak_weeks
FROM StreakGroups
GROUP BY user_id
ORDER BY longest_streak_weeks DESC
LIMIT 1;
"""
df_11 = pd.read_sql(query_11, engine)
print(df_11)

   users_with_4plus_week_streak user_id  longest_streak_weeks
0                          1675  U00213                    26


## Q12 - Churn signal detection
**Answer:** Identification of users with >= 50% drop in watch minutes from May 2024 to June 2024.

In [63]:
query_12 = """
WITH MonthlyMinutes AS (
    SELECT
        user_id,
        SUM(CASE WHEN MONTH(session_date) = 5 THEN watch_minutes ELSE 0 END) AS may_mins,
        SUM(CASE WHEN MONTH(session_date) = 6 THEN watch_minutes ELSE 0 END) AS june_mins
    FROM watch_sessions
    WHERE user_id IS NOT NULL
      AND YEAR(session_date) = 2024
      AND MONTH(session_date) IN (5, 6)
    GROUP BY user_id
),
ChurnList AS (
    SELECT
        m.user_id,
        u.name,
        m.may_mins,
        m.june_mins,
        ROUND(((m.may_mins - m.june_mins) * 100.0) / m.may_mins, 2) AS drop_percentage
    FROM MonthlyMinutes m
    JOIN users u ON m.user_id = u.user_id
    WHERE m.may_mins > 0
      AND ((m.may_mins - m.june_mins) * 1.0) / m.may_mins >= 0.5
)
SELECT
    (SELECT COUNT(*) FROM ChurnList) AS total_churn_signals,
    user_id,
    name,
    may_mins,
    june_mins,
    drop_percentage
FROM ChurnList
ORDER BY drop_percentage DESC;
"""
df_12 = pd.read_sql(query_12, engine)
print(f"Total Churn Signal Users: {df_12['total_churn_signals'].iloc[0] if not df_12.empty else 0}\n")
print(df_12.head(10))

Total Churn Signal Users: 521

   total_churn_signals user_id              name  may_mins  june_mins  \
0                  521  U00023    Amit Mukherjee      43.0        0.0   
1                  521  U00166  Shaurya Malhotra      94.0        0.0   
2                  521  U00211        Ravi Menon     336.0        0.0   
3                  521  U00225     Kritika Patil     209.0        0.0   
4                  521  U00237    Shaurya Bansal     126.0        0.0   
5                  521  U00262        Suresh Roy     158.0        0.0   
6                  521  U00271       Krish Patel     227.0        0.0   
7                  521  U00282        Tara Kumar      47.0        0.0   
8                  521  U00289    Rohan Krishnan      18.0        0.0   
9                  521  U00292    Kavitha Kamath     594.0        0.0   

   drop_percentage  
0            100.0  
1            100.0  
2            100.0  
3            100.0  
4            100.0  
5            100.0  
6            100.0